# Book Endings

## Part 1
You are reading a Build Your Own Story book. It is like a normal book except that choices on some pages affect the story, sending you to one of two options for your next page.

These choices are really stressing you out, so you decide that you'll either always pick the first option, or always pick the second option.

You start reading at page 1 and read forward one page at a time unless you reach a choice or an ending.

The choices are provided in a list, sorted by the page containing the choice, and each choice has two options of pages to go to next. In this example, you are on page 3, where there is a choice. Option 1 goes to page 14 and Option 2 goes to page 2.

choices1 = [[3, 14, 2]] => [current_page, option_1, option_2] The ending pages are also given in a sorted list:

endings = [6, 15, 21, 30]

Given a list of endings, a list of choices with their options, and the choice you will always take (Option 1 or Option 2), return the ending you will reach. If you get stuck in a loop, return -1.

### Approach 
* basically traversing a decision tree until I reach a root node (ending)
* using a hash map to keep track of visited pages, so I know when I'm in a loop

In [ ]:
def find_ending(endings, choices, option):
    current_choices = 0 
    visited = {}
    page = 1
    while page not in endings or page > endings[-1]:
        if current_choices < len(choices)-1 and page > choices[current_choices][0]:
            current_choices += 1
            continue
        if current_choices < len(choices) and page == choices[current_choices][0]:
            next_page = choices[current_choices][option]
            current_choices = 0
        else:
            next_page = page+1

        if visited.get(next_page):
            return -1
        visited[page] = next_page       
        page = next_page   
    return page
    
endings = [6, 15, 21, 30]
choices1 = [[3, 14, 2]]
choices2 = [[5, 11, 28], [9, 19, 29], [14, 16, 20], [18, 7, 22], [25, 6, 30]]
choices3 = []
choices4 = [[2, 10, 15], [3, 4, 10], [4, 3, 15], [10, 3, 15]]



print(find_ending(endings, choices1, 1) == 15)
print(find_ending(endings, choices1, 2) == -1)
print(find_ending(endings, choices2, 1) == 21)
print(find_ending(endings, choices2, 2) == 30)
print(find_ending(endings, choices3, 1) == 6)
print(find_ending(endings, choices3, 2) == 6)
print(find_ending(endings, choices4, 1) == -1)
print(find_ending(endings, choices4, 2) == 15)

### Analysis
* Time Complexity: O(N+C) (usual case)
    * N: number of pages to traverse, worst case would be N = endings[-1]
    * C: length of choices
* Unusual case Time Complexity: O(N*C)
    * happens if we end up having to reset `current_choices` to 0 constantly, like if we did it at almost every page we visit
* Space Complexity: O(N)
    * worst case: you'd have to store every page in the `visited` dictionary

## Part 2
As you are reading the book multiple times, you notice that you always get bad endings. You start to suspect there is no way to get to the good endings, so you decide to find out.

Good and bad endings will be separate lists of page numbers, like this:

good_endings = [10, 15, 25, 34] bad_endings = [21, 30, 40]

Given lists of good endings, bad endings, and a list of the choices along with their options, return a collection of all the reachable good endings, if any.

### Approach
* I will probably use recursion with a tree like structure to traverse the possible endings to find all the good endings
* I will do a depth first search on all possible paths
* returning a set to simplify getting rid of duplicate values in my collection of all reachable good endings

In [39]:
def find_good_endings(good_endings, bad_endings, choices, page = 1, visited = None):
    if visited == None:
        visited = {}
    if page in good_endings:
        return [page]
    if page in bad_endings:
        return []
    if page > good_endings[-1] or page > bad_endings[-1]:
        return []
    if visited.get(page):
        return []
    reachable = []
    current_choice = 0
    while current_choice < len(choices) and page != choices[current_choice][0]:
        current_choice += 1

    if current_choice < len(choices) and page == choices[current_choice][0]:
        option1 = choices[current_choice][1]
        option2 = choices[current_choice][2]
        visited[page] = [option1, option2]
        reachable += find_good_endings(good_endings, bad_endings, choices, page=option1, visited=visited)
        reachable += find_good_endings(good_endings,bad_endings,choices, page=option2, visited=visited)

    else:
        next_page = page + 1
        visited[page] = next_page
        reachable +=  find_good_endings(good_endings,bad_endings, choices, page=next_page, visited=visited)
    visited = {}
    return set(reachable)

good_endings = [10, 15, 25, 34]
bad_endings = [21, 30, 40]
choices1 = [[3, 16, 24]]
choices2 = [[3, 16, 20]]
choices3 = [[3, 2, 19], [20, 21, 34]]
choices4 = []
choices5 = [[9, 16, 26], [14, 16, 13], [27, 29, 28], [28, 15, 34], [29, 30, 38]]
choices6 = [[9, 16, 26], [13, 31, 14], [14, 16, 13], [27, 12, 24], [32, 34, 15]]
choices7 = [[3, 9, 10]]

print(find_good_endings(good_endings, bad_endings, choices1)) #== [25])
print(find_good_endings(good_endings, bad_endings, choices2)) #== [])
print(find_good_endings(good_endings, bad_endings, choices3)) #== [34])
print(find_good_endings(good_endings, bad_endings, choices4)) #== [10])
print(find_good_endings(good_endings, bad_endings, choices5)) #== [15, 34])
print(find_good_endings(good_endings, bad_endings, choices6)) #== [15, 25, 34])
print(find_good_endings(good_endings, bad_endings, choices7)) #== [10])

{25}
set()
{34}
{10}
{34, 15}
{25, 34, 15}
{10}


## Part 3
You really love this book and so you decide to read all the story sequences. You notice that you are reading some pages more than others. You want to find out which page you read the most often when you read every storyline.

You set some rules for your reading to avoid repeating pages too often. These rules are:

All storylines start at page 1.
In one storyline you will never make the same choice twice (you can choose the other option)
If you reach a choice where you've already made both choices, this doesn't count as a valid storyline (all storylines conclude at an ending).
Given a list of endings and a list of the choices with destinations, return the page which was read the most often and how many times it was read. If multiple pages were read the same number of times, return any of them. If there are no valid storylines, return -1 or None.

In [ ]:
endings_1 = [5, 10]  
choices1_1 = [[3, 7, 9], [9, 10, 8], [9, 10, 8]]

